Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [2]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

model=qwen3.8-flash


## Which of these needs an agent?

- An agent costs a model call per step, takes seconds, and can be wrong
- A chain costs one call and always runs the same path
- Ordinary code costs nothing, is instant, and is right every time
- The skill is telling them apart before anyone starts building

Five tasks below, all real ones from operations and support work. Decide for
each: `code`, `one model call`, or `agent`. Write our answer in the cell
under each, then run the cell after it and see what happens.

There is no marking. The reasoning is the exercise.

### Task 1

*"Every night, take yesterday's tickets and produce a count per category. The
categories are fixed and each ticket already has one."*

Our answer :

In [3]:
answer_1 = "code / one call / agent"   # <- replace with your choice

### Task 2

*"A customer writes in. Decide whether it goes to access, billing or field, and
draft a first reply."*

Our answer :

In [4]:
answer_2 = "code / one call / agent"

### Task 3

*"Find out why the Rotterdam depot's numbers looked wrong last month. We may
need the incident log, the shift roster and the maintenance record ; we will
not know which until we start."*

Our answer :

In [5]:
answer_3 = "code / one call / agent"

### Task 4

*"Rewrite this refund refusal so it is polite but does not admit fault."*

Our answer :

In [6]:
answer_4 = "code / one call / agent"

### Task 5

*"Check whether an IBAN in a message is formatted correctly."*

Our answer :

In [7]:
answer_5 = "code / one call / agent"

## What the course would say

Run this. It is not a marking scheme, it is an argument we can disagree with.

In [8]:
VERDICT = {
 1: ("code",
     "Fixed rule, fixed categories, no judgement. A model would cost a call per "
     "ticket and occasionally miscount. This is a GROUP BY."),
 2: ("agent",
     "Routing needs judgement, and drafting needs the customer record, so it "
     "has to look something up before it can answer. Two steps, second depends "
     "on the first."),
 3: ("agent",
     "The clearest case on the page. Nobody can write the steps in advance, "
     "because which source matters depends on what the last one said."),
 4: ("one call",
     "Judgement, but one shot. Nothing to look up and nothing to decide about "
     "what to do next. An agent here is a chain wearing a costume."),
 5: ("code",
     "IBAN has a checksum. A regular expression and mod-97 answers it exactly, "
     "for free, forever. A model will be right most of the time, which is worse "
     "than a rule that is right every time."),
}

for n in range(1, 6):
    mine = globals().get(f"answer_{n}", "").strip().lower()
    want, why = VERDICT[n]
    if mine == "code / one call / agent" or not mine:
        mark = "you left this one blank"
    elif mine == want:
        mark = "same as yours"
    else:
        mark = f"you said {mine}"
    print(f"Task {n}: course says {want.upper():9} - {mark}")
    print(f"         {why}\n")

Task 1: course says CODE      - you left this one blank
         Fixed rule, fixed categories, no judgement. A model would cost a call per ticket and occasionally miscount. This is a GROUP BY.

Task 2: course says AGENT     - you left this one blank
         Routing needs judgement, and drafting needs the customer record, so it has to look something up before it can answer. Two steps, second depends on the first.

Task 3: course says AGENT     - you left this one blank
         The clearest case on the page. Nobody can write the steps in advance, because which source matters depends on what the last one said.

Task 4: course says ONE CALL  - you left this one blank
         Judgement, but one shot. Nothing to look up and nothing to decide about what to do next. An agent here is a chain wearing a costume.

Task 5: course says CODE      - you left this one blank
         IBAN has a checksum. A regular expression and mod-97 answers it exactly, for free, forever. A model will be right most

## The pattern

Look back at the three the course calls `code` or `one call`. In each, the
steps were knowable in advance. In the two it calls `agent`, they were not :
something had to be looked at before the next step could be chosen.

That is the whole test, and it is worth more than any framework :

> Can we write down the steps before we start? Then write them down.
> Does step two depend on what step one found? Then we may want an agent.

Everything else, cost, latency, the chance of a confident wrong answer, makes
the bar higher, never lower.

### Try it on our own work

Not code. Pick one task we actually do, or one we have been asked whether
"AI could do", and answer four questions about it :

1. Can we write the steps down in advance? If yes, stop ; it is not an agent
2. What would it need to look at, and does that exist as something a program
   could read?
3. What is the worst wrong answer it could give, and who would notice?
4. Which step would we want a person to approve before it takes effect?

If we can answer those four, we can brief a developer or judge a vendor.
That is further than most people who ask for an agent ever get.